# Clustering and PCA — Concepts

Read the idea, the formula and the small example in that order. The demonstration code is provided.

## Libraries and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

## 1. Unsupervised learning has no target

Regression and classification learn from inputs **and known answers**. Clustering and PCA use inputs alone.

**Clustering** groups similar observations. **PCA** summarises several inputs using fewer new axes. There is no survival or price target for training here, so accuracy or supervised test RMSE is not the appropriate evaluation. We examine separation, stability and whether the result is useful to interpret. External labels, if available, can help assess results afterwards.

## 2. k-means: assign, average, repeat

1. Choose the number of clusters $k$.
2. Place $k$ initial centres.
3. Assign each observation to its nearest centre.
4. Move each centre to the mean of its observations; repeat steps 3–4 until little changes.

The objective is small within-cluster squared distance:
$$\mathrm{WCSS}=\sum_{c=1}^{k}\sum_{x_i\in C_c}\lVert x_i-\mu_c\rVert^2.$$
$C_c$ is cluster c and $\mu_c$ its centre. $\lVert x_i-\mu_c\rVert^2$ means the sum of squared differences across inputs. For points (1,1) and (3,1), the centre is (2,1) and WCSS is $1+1=2$.

`KMeans(n_clusters=3, n_init=10, random_state=42)` tries ten initialisations and keeps the smallest WCSS. Starting centres can affect the result, so use several starts. Scale features when their units differ.

In [ ]:
points = np.array([[1, 1], [3, 1]])
centre = points.mean(axis=0)
print('Centre:', centre)
print('WCSS:', np.sum((points - centre) ** 2))

## 3. Choose k with an elbow plot

Plot WCSS (`inertia_`) against k. More clusters can always reduce the optimal WCSS; with one cluster per observation it is zero. Look for a bend where extra clusters bring smaller improvements. The elbow is a guide, not proof of one correct k.

**Interpretation:** examine cluster centres, sizes and observations. **Stability:** different starts should give broadly similar groups; `n_init=10` helps. Cluster numbers are arbitrary: cluster 0 is not a rank.

An optional **silhouette score** compares within-cluster distance $a$ with nearest-other-cluster distance $b$:
$$s=\frac{b-a}{\max(a,b)}.$$
A value near 1 suggests good separation; near 0, overlap; negative values suggest the point might fit another cluster better. You do not need to program silhouette analysis for this exercise.

## 4. Hierarchical clustering and a dendrogram

Agglomerative clustering begins with one cluster per observation. It repeatedly merges two clusters until all observations belong to one cluster. A **dendrogram** shows this merge history.

The height of a merge measures separation according to the chosen method. With **Ward linkage**, merges are chosen to keep the increase in within-cluster squared variation small. Draw a horizontal cut: the number of branches it crosses gives the number of clusters. Long vertical branches can suggest a useful cut. Scale before clustering; do not compare merge heights across incompatible scales.

In [ ]:
points = np.array([[1, 1], [1.2, 1], [4, 4], [4.2, 4], [8, 1], [8.2, 1]])
Z = linkage(points, method='ward')
dendrogram(Z, labels=['A', 'B', 'C', 'D', 'E', 'F'])
plt.axhline(3, color='red', linestyle='--', label='Example cut')
plt.ylabel('Ward linkage height')
plt.legend()
plt.show()

## 5. PCA: a compact description

PCA rotates the coordinate system. **PC1** captures as much spread (variance) as possible. **PC2** captures the most remaining spread, at right angles to PC1. PCA does not create clusters.

A component is a weighted combination of centred inputs; for example:
$$\mathrm{PC1}=a_1z_1+a_2z_2+a_3z_3.$$
$z_j$ are standardised inputs and $a_j$ are weights (**loadings**, in `components_`). Explained variance describes how much spread a component retains:
$$\mathrm{explained\ variance\ ratio}_j=\frac{\lambda_j}{\sum_l\lambda_l}.$$
$\lambda_j$ is the variance along component j. If PC1 retains 88%, one axis summarises most of the variation. PCA has no target; explained variance is not prediction accuracy. Standardise inputs with different units first. We use PCA as a short outlook.

### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [ ]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))